# ComfyUI + Z-Image Turbo - 2560x1440 生成

Google Colabで高解像度画像を生成します

## 手順
1. ランタイム → GPU T4以上を選択
2. すべてのセルを実行
3. 生成完了後、画像をダウンロード

In [ ]:
# GPU確認
import torch
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"VRAM: {vram:.1f} GB")

In [ ]:
# ComfyUIインストール
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd ComfyUI
!pip install -r requirements.txt -q

In [ ]:
# モデルダウンロード（別のソースから）
import os
import subprocess

models_dir = "/content/ComfyUI/models/checkpoints"
os.makedirs(models_dir, exist_ok=True)

# 方法1: Comfy-Org版（FP16、12GB）- こちらを試す
print("Downloading from Comfy-Org...")
subprocess.run([
    "wget", "-q", "--show-progress",
    "-O", f"{models_dir}/z-image-turbo-fp16.safetensors",
    "https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/diffusion_models/z_image_turbo_bf16.safetensors"
], check=True)
print("Download complete!")

In [ ]:
# テキストエンコーダー（Qwen 3B）
print("Downloading text encoder...")
subprocess.run([
    "wget", "-q", "--show-progress",
    "-O", "/content/ComfyUI/models/text_encoders/qwen_3_4b.safetensors",
    "https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors"
], check=True)
print("Text encoder downloaded")

In [ ]:
# VAE
print("Downloading VAE...")
subprocess.run([
    "wget", "-q", "--show-progress",
    "-O", "/content/ComfyUI/models/vae/ae.safetensors",
    "https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors"
], check=True)
print("VAE downloaded")

In [ ]:
# configファイル
os.makedirs("/content/ComfyUI/models/configs", exist_ok=True)
subprocess.run(["wget", "-q", "-O", 
    "/content/ComfyUI/models/configs/v1-inference.yaml",
    "https://raw.githubusercontent.com/comfyanonymous/ComfyUI/master/configs/v1-inference.yaml"
], check=True)
print("Config OK")

In [ ]:
# ファイル名確認
import os
print("Checkpoints:")
print(os.listdir("/content/ComfyUI/models/checkpoints/"))
print("\nText Encoders:")
print(os.listdir("/content/ComfyUI/models/text_encoders/"))
print("\nVAE:")
print(os.listdir("/content/ComfyUI/models/vae/"))

In [ ]:
# ComfyUI起動（別スレッド）
import threading
import subprocess

def run_comfyui():
    subprocess.run(["python", "main.py", "--listen", "0.0.0.0", "--port", "8188"])

t = threading.Thread(target=run_comfyui, daemon=True)
t.start()
print("ComfyUI starting...")
import time
time.sleep(10)
print("ComfyUI should be running")

In [ ]:
# 接続確認
import requests
try:
    r = requests.get("http://127.0.0.1:8188/system_stats")
    print("ComfyUI connected!")
except:
    print("Waiting for ComfyUI...")
    import time
    time.sleep(20)
    r = requests.get("http://127.0.0.1:8188/system_stats")
    print("Connected!")

In [ ]:
# 画像生成（2560x1440）
import requests
import time

# モデルファイル名を確認して設定
model_file = "z_image_turbo_bf16.safetensors"

workflow = {
    "1": {"inputs": {"model_name": model_file}, "class_type": "DiffusionLoader"},
    "2": {"inputs": {"clip_name": "qwen_3_4b.safetensors"}, "class_type": "CLIPLoader"},
    "3": {"inputs": {"vae_name": "ae.safetensors"}, "class_type": "VAELoader"},
    "4": {"inputs": {"text": "Professional stock photo of Japanese businessman in suit walking through Tokyo Shibuya crossing at night, neon lights, crowd, cinematic lighting, 4K quality", "clip": ["2", 0]}, "class_type": "CLIPTextEncode"},
    "5": {"inputs": {"text": "", "clip": ["2", 0]}, "class_type": "CLIPTextEncode"},
    "6": {"inputs": {"width": 2560, "height": 1440, "batch_size": 1}, "class_type": "EmptyLatentImage"},
    "7": {"inputs": {"model": ["1", 0], "seed": 42, "steps": 8, "cfg": 1.0, "sampler_name": "euler", "scheduler": "simple", "positive": ["4", 0], "negative": ["5", 0], "latent_image": ["6", 0]}, "class_type": "KSampler"},
    "8": {"inputs": {"samples": ["7", 0], "vae": ["3", 0]}, "class_type": "VAEDecode"},
    "9": {"inputs": {"images": ["8", 0], "filename_prefix": "result_2560x1440"}, "class_type": "SaveImage"}
}

print("Generating 2560x1440 image...")
result = requests.post("http://127.0.0.1:8188/prompt", json={"prompt": workflow}).json()
print(f"Result: {result}")

if "prompt_id" in result:
    prompt_id = result['prompt_id']
    
    for i in range(300):
        time.sleep(1)
        hist = requests.get(f"http://127.0.0.1:8188/history/{prompt_id}").json()
        if prompt_id in hist and hist[prompt_id]['status']['completed']:
            print("Complete!")
            break
        if i % 30 == 0:
            print(f"Progress: {i}s...")
else:
    print("Error: " + str(result))

In [ ]:
# 画像ダウンロード
from google.colab import files
import glob

output_files = glob.glob("/content/ComfyUI/output/result_2560x1440_*.png")
print(f"Found: {output_files}")

if output_files:
    latest = max(output_files, key=os.path.getmtime)
    print(f"Downloading: {latest}")
    files.download(latest)
else:
    print("No output found - check ComfyUI console")